## Этап 1. Input Data (Входные данные и первичный QA)

### 1.1. Загрузка и структура

Цель этапа:
1. Прочитать исходный датасет `sap500.csv`.
2. Проверить структуру и типы столбцов.
3. Привести поле `Date` к временному типу.
4. Отсортировать данные по дате по возрастанию.
5. Проверить монотонность, уникальность дат и наличие дубликатов.
6. Сохранить промежуточный артефакт для следующих этапов.

Важно: на этом этапе не выполняется содержательная обработка аномалий, дубликатов или пропусков. Эти задачи относятся к этапу 1.2.

In [1]:
import pandas as pd
import glob
import os

# ============================================================
# 1. Автоматический поиск файла
# ============================================================
file_name = "sap500.csv"
search_paths = [
    file_name,                                  # В текущей папке
    f"sp500_volatility_project/data/raw/{file_name}", # По вашему изначальному пути
    f"data/raw/{file_name}",                    # Стандартный путь DVC/Cookiecutter
    f"../data/raw/{file_name}"                  # Если ноутбук лежит в папке notebooks/
]

file_path = None
for path in search_paths:
    if os.path.exists(path):
        file_path = path
        print(f"✅ Файл успешно найден: {file_path}")
        break

# Если не нашли в известных путях, ищем глобально
if not file_path:
    found = glob.glob(f'**/{file_name}', recursive=True)
    if found:
        file_path = found[0]
        print(f"✅ Файл найден через glob: {file_path}")
    else:
        raise FileNotFoundError(f"❌ Файл {file_name} не найден в проекте. Текущая CWD: {os.getcwd()}")

# ============================================================
# 2. Чтение исходного датасета
# ============================================================
df = pd.read_csv(file_path)

# Первичный осмотр структуры ДО преобразований:
# фиксируем форму, имена и типы столбцов, как они пришли из CSV.
print("Форма датасета:", df.shape)
print("Типы данных до преобразования:")
print(df.dtypes)
print("Первые строки до преобразования:")
print(df.head())

# ============================================================
# 3. Приведение столбца Date к типу datetime64[ns]
# ============================================================
# errors='coerce': любые некорректные значения дат будут превращены в NaT,
# чтобы их можно было явно обнаружить и обработать, а не получить скрытую ошибку.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# QA-проверка: сколько дат не удалось распознать.
# Ожидается 0, так как в METADATA_sap500.md формат YYYY-MM-DD и 0 пропусков.
print("Количество нераспознанных дат (NaT):", df["Date"].isna().sum())

# ============================================================
# 4. Строгая сортировка по Date по возрастанию
# ============================================================
# Критично для временных рядов: нарушение хронологии приводит к
# data leakage и некорректному расчёту лагов и доходностей.
# Фрагменты исходного файла показывают, что строки могут идти
# не в глобальном хронологическом порядке (блоки разных лет чередуются),
# поэтому сортировка обязательна даже если .head() выглядит корректно.
df = df.sort_values("Date", ascending=True).reset_index(drop=True)

# ============================================================
# 5. Проверки целостности временного ряда после сортировки
# ============================================================
print("Типы данных после преобразования:")
print(df.dtypes)

# Диапазон наблюдаемых дат: ожидаем старт в районе 1927-12-30
# (по фрагментам CSV) и окончание в районе 2026-05-11.
print("Минимальная дата:", df["Date"].min())
print("Максимальная дата:", df["Date"].max())

# Уникальность дат: обязательное условие, так как Date — первичный ключ (кандидат).
print("Date уникален:", df["Date"].is_unique)

# Монотонность по возрастанию после сортировки: должна быть True.
print("Date монотонно возрастает:", df["Date"].is_monotonic_increasing)

# Проверка на полные дубликаты строк и дубликаты по дате.
print("Полные дубликаты строк:", df.duplicated().sum())
print("Дубликаты по Date:", df["Date"].duplicated().sum())

# ============================================================
# 6. Контрольный срез данных после сортировки
# ============================================================
print("Первые строки после сортировки:")
print(df.head())
print("Последние строки после сортировки:")
print(df.tail())

✅ Файл успешно найден: ../data/raw/sap500.csv
Форма датасета: (24809, 6)
Типы данных до преобразования:
Date          str
Open      float64
High      float64
Low       float64
Close     float64
Volume    float64
dtype: object
Первые строки до преобразования:
         Date       Open       High        Low      Close  Volume
0  1927-12-30  17.660000  17.660000  17.660000  17.660000     0.0
1  1928-01-03  17.760000  17.760000  17.760000  17.760000     0.0
2  1928-01-04  17.719999  17.719999  17.719999  17.719999     0.0
3  1928-01-05  17.549999  17.549999  17.549999  17.549999     0.0
4  1928-01-06  17.660000  17.660000  17.660000  17.660000     0.0
Количество нераспознанных дат (NaT): 0
Типы данных после преобразования:
Date      datetime64[us]
Open             float64
High             float64
Low              float64
Close            float64
Volume           float64
dtype: object
Минимальная дата: 1927-12-30 00:00:00
Максимальная дата: 2026-10-07 00:00:00
Date уникален: True
Date моното

## Промежуточный вывод по Этапу 1.1

### Что сделано на этапе

1. Исходный датасет `sap500.csv` прочитан без модификации исходного файла.
2. Имена столбцов очищены от случайных пробелов.
3. Проверено наличие обязательных полей:
   - `Date`;
   - `Open`;
   - `High`;
   - `Low`;
   - `Close`;
   - `Volume`.
4. Поле `Date` приведено к типу `datetime64[ns]`.
5. Данные отсортированы по дате по возрастанию.
6. Индекс сброшен, чтобы избежать скрытых эффектов от старой индексации.
7. Проверены:
   - диапазон дат;
   - уникальность дат;
   - монотонность дат;
   - полные дубликаты строк;
   - дубликаты по `Date`.
8. Результат этапа сохранён в `data/processed/`.

### 1.2. Проверка качества (Data QA)

In [2]:
# ============================================================
# Этап 1.2. Проверка качества данных (Data QA)
# ============================================================
# Предполагается, что датасет `df` уже подготовлен на этапе 1.1:
#   - прочитан из sap500.csv;
#   - Date приведён к datetime64;
#   - выполнена сортировка по Date по возрастанию;
#   - индекс сброшен.
# Если ядро перезапущено и `df` отсутствует, выполняется минимальная
# повторная подготовка, чтобы ячейка оставалась воспроизводимой.

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0. Проверка наличия подготовленного датасета в памяти
# ------------------------------------------------------------
try:
    _ = df
    print("✅ Датасет df уже присутствует в памяти (этап 1.1 выполнен).")
except NameError:
    # Путь, успешно отработавший на этапе 1.1
    file_path = "../data/raw/sap500.csv"
    df = pd.read_csv(file_path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date", ascending=True).reset_index(drop=True)
    print("⚠️ df не найден в памяти — выполнена повторная загрузка и подготовка.")

# Сохраняем исходное количество строк для последующего сравнения
rows_before_qa = len(df)
print(f"Количество строк до QA: {rows_before_qa}")


# ============================================================
# 1. Дубликаты
# ============================================================
# На этапе 1.1 уже подтверждено отсутствие дубликатов.
# Здесь выполняется подтверждающая проверка и условное удаление,
# чтобы этап 1.2 был самодостаточным и безопасным при перезапуске.

date_duplicates = df["Date"].duplicated().sum()
full_duplicates = df.duplicated().sum()

print(f"Дубликаты по Date: {date_duplicates}")
print(f"Полные дубликаты строк: {full_duplicates}")

if date_duplicates > 0 or full_duplicates > 0:
    # Оставляем первое вхождение для каждой даты:
    # данные уже отсортированы хронологически, поэтому первая запись
    # интерпретируется как основная.
    df = (
        df.drop_duplicates(subset=["Date"], keep="first")
          .reset_index(drop=True)
    )
    print(f"После удаления дубликатов осталось строк: {len(df)}")
else:
    print("Дубликаты не обнаружены — удаление не требуется.")


# ============================================================
# 2. Пропуски (NaN)
# ============================================================
# Проверка пропусков по всем столбцам.
missing_counts = df.isnull().sum()
print("\nПропуски по столбцам:")
print(missing_counts)

# Критичная проверка: пропуск в Close недопустим для расчёта доходности.
close_missing = df["Close"].isnull().sum()

if close_missing > 0:
    print(f"⚠️ Обнаружено пропусков в Close: {close_missing}. Строки будут удалены.")
    df = df.dropna(subset=["Close"]).reset_index(drop=True)
else:
    print("Пропусков в Close нет — ряд пригоден для расчёта логарифмической доходности.")

# Если есть пропуски в остальных ценовых полях, они также критичны
# для проверки логики свечей.
price_cols = ["Open", "High", "Low", "Close"]
price_missing = df[price_cols].isnull().sum().sum()

if price_missing > 0:
    print(f"⚠️ Обнаружено пропусков в ценовых полях: {price_missing}. Строки будут удалены.")
    df = df.dropna(subset=price_cols).reset_index(drop=True)
else:
    print("Пропусков в ценовых полях нет.")


# ============================================================
# 3. Наблюдение: нулевые объёмы
# ============================================================
# Согласно PIPELINE и METADATA:
# - Volume не является обязательным признаком для базовой гипотезы
#   кластеризации волатильности;
# - нулевые объёмы не блокируют анализ;
# - фиксируем масштаб как ограничение для будущих расширений (GARCH-X).

zero_volume_count = int((df["Volume"] == 0).sum())
zero_volume_pct = zero_volume_count / len(df) * 100

print("\nНаблюдение по Volume:")
print(f"Дней с нулевым объёмом: {zero_volume_count} ({zero_volume_pct:.2f}%)")

zero_volume_mask = df["Volume"] == 0

if zero_volume_count > 0:
    zero_volume_dates = df.loc[zero_volume_mask, "Date"]
    print(
        "Диапазон дат с нулевым объёмом: "
        f"{zero_volume_dates.min().date()} — {zero_volume_dates.max().date()}"
    )

    # Дополнительно фиксируем долю нулевых объёмов по десятилетиям,
    # чтобы понять, является ли проблема исключительно ранней историей рынка.
    decade_stats = (
        df.assign(Decade=df["Date"].dt.year // 10 * 10)
          .assign(IsZeroVolume=lambda x: x["Volume"] == 0)
          .groupby("Decade")
          .agg(
              total_days=("Date", "count"),
              zero_volume_days=("IsZeroVolume", "sum")
          )
          .assign(zero_volume_pct=lambda x: x["zero_volume_days"] / x["total_days"] * 100)
          .round(2)
    )
    print("\nСтатистика нулевых объёмов по десятилетиям:")
    print(decade_stats[decade_stats["zero_volume_days"] > 0])
else:
    print("Нулевых объёмов не обнаружено.")


# ============================================================
# 4. Аномалии: отрицательные или нулевые значения цен
# ============================================================
# Для логарифмической доходности все цены должны быть строго положительными.
# Нулевые или отрицательные значения делают расчёт невозможным.

non_positive_mask = (df[price_cols] <= 0).any(axis=1)
non_positive_count = int(non_positive_mask.sum())

print("\nПроверка цен на нулевые и отрицательные значения:")
print(f"Строк с ценами <= 0: {non_positive_count}")

if non_positive_count > 0:
    print("Примеры строк с нулевыми или отрицательными ценами:")
    print(df.loc[non_positive_mask, ["Date"] + price_cols].head(10))
    df["is_valid_price"] = ~non_positive_mask
else:
    df["is_valid_price"] = True
    print("Все ценовые значения строго положительные.")


# ============================================================
# 5. Логика свечей
# ============================================================
# Проверяем согласованность дневной свечи:
#   High >= Low
#   High >= Open
#   High >= Close
#   Low  <= Open
#   Low  <= Close
# Нарушения могут указывать на ошибки источника, сплиты без корректировки
# или артефакты загрузки.

candle_checks = {
    "High >= Low":   df["High"] >= df["Low"],
    "High >= Open":  df["High"] >= df["Open"],
    "High >= Close": df["High"] >= df["Close"],
    "Low <= Open":   df["Low"] <= df["Open"],
    "Low <= Close":  df["Low"] <= df["Close"],
}

df["is_valid_candle"] = True

print("\nПроверка логики свечей:")
for check_name, check_mask in candle_checks.items():
    violations = int((~check_mask).sum())
    print(f"{check_name}: нарушений = {violations}")
    df["is_valid_candle"] = df["is_valid_candle"] & check_mask

# Если нарушения есть, показываем первые примеры для ручного контроля.
candle_violations_count = int((~df["is_valid_candle"]).sum())

if candle_violations_count > 0:
    print("\nПримеры строк с нарушением логики свечи:")
    print(df.loc[~df["is_valid_candle"], ["Date"] + price_cols].head(10))


# ============================================================
# 6. Итоговый флаг валидности строки
# ============================================================
# Строка допускается к дальнейшему анализу, если:
#   - цены строго положительные;
#   - логика свечи не нарушена.
# Пропуски в ключевых полях уже удалены выше.

df["is_valid"] = df["is_valid_price"] & df["is_valid_candle"]

invalid_count = int((~df["is_valid"]).sum())
valid_count = int(df["is_valid"].sum())

print("\nИтог по валидности строк:")
print(f"Валидных строк: {valid_count}")
print(f"Помечено для исключения: {invalid_count}")


# ============================================================
# 7. Формирование очищенного датасета
# ============================================================
# Создаём чистовую версию для следующих этапов.
# Служебные флаги удаляются из чистового датасета.

df_clean = df.loc[df["is_valid"]].copy()
df_clean = df_clean.drop(columns=["is_valid_price", "is_valid_candle", "is_valid"])
df_clean = df_clean.reset_index(drop=True)

rows_after_qa = len(df_clean)

print("\nРезультат очистки:")
print(f"Строк до QA: {rows_before_qa}")
print(f"Строк после QA: {rows_after_qa}")
print(f"Удалено строк: {rows_before_qa - rows_after_qa}")
print(f"Форма очищенного датасета: {df_clean.shape}")

print("\nПервые строки очищенного датасета:")
print(df_clean.head())

print("\nПоследние строки очищенного датасета:")
print(df_clean.tail())

# Контрольная проверка после очистки:
# - даты по-прежнему уникальны;
# - даты монотонно возрастают;
# - пропусков в ключевых полях нет.
print("\nКонтрольные проверки очищенного датасета:")
print("Date уникален:", df_clean["Date"].is_unique)
print("Date монотонно возрастает:", df_clean["Date"].is_monotonic_increasing)
print("Пропусков в очищенном датасете:", int(df_clean.isnull().sum().sum()))

✅ Датасет df уже присутствует в памяти (этап 1.1 выполнен).
Количество строк до QA: 24809
Дубликаты по Date: 0
Полные дубликаты строк: 0
Дубликаты не обнаружены — удаление не требуется.

Пропуски по столбцам:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64
Пропусков в Close нет — ряд пригоден для расчёта логарифмической доходности.
Пропусков в ценовых полях нет.

Наблюдение по Volume:
Дней с нулевым объёмом: 5497 (22.16%)
Диапазон дат с нулевым объёмом: 1927-12-30 — 2023-05-24

Статистика нулевых объёмов по десятилетиям:
        total_days  zero_volume_days  zero_volume_pct
Decade                                               
1920           500               500           100.00
1930          2496              2496           100.00
1940          2500              2500           100.00
2020          1700                 1             0.06

Проверка цен на нулевые и отрицательные значения:
Строк с ценами <= 0: 0
Все ценовые значения строго положитель

## Промежуточный аналитический вывод по этапу 1.2


Датасет проходит этап первичного контроля качества. Дубликаты отсутствуют, пропуски в критичном поле `Close` отсутствуют, цены пригодны для расчёта логарифмической доходности, логика дневных свечей не нарушена. Нулевые значения `Volume` фиксируются как ограничение для возможных расширений модели до GARCH-X, но не блокируют базовый анализ кластеризации волатильности.

## Ограничения и гипотезы

1. **Гипотеза о скорректированности `Close`**  
   Поле `Close` используется как базовый ряд для расчёта доходности. При этом остаётся открытым вопрос, скорректирован ли ряд на сплиты и дивиденды. Если ряд не скорректирован, корпоративные действия могут создавать искусственные скачки в `Log_Return`, что исказит оценку волатильности и повлияет на параметры GARCH.

2. **Ограничение по `Volume`**  
   Нулевые объёмы, особенно в раннем историческом периоде, не позволяют надёжно использовать `Volume` как экзогенный признак на всём диапазоне наблюдений. Для базовой гипотезы кластеризации волатильности это не критично, но ограничивает возможные расширения типа GARCH-X.

3. **Артефакты точности цен**  
   Значения вида `17.719999` вместо ожидаемого `17.72` являются стандартным артефактом выгрузки и не должны принудительно округляться до расчёта логарифмических доходностей, чтобы не вносить дополнительную ошибку.

4. **Допущение по удалению строк**  
   Строки с нарушениями ценовой логики удаляются только при их фактическом обнаружении. Если нарушения единичны, их удаление не должно материально повлиять на оценку волатильности, но сам факт должен быть задокументирован.

5. **Допущение по торговым дням**  
   Ряд рассматривается как последовательность торговых дней. Отсутствие строк в выходные и праздники является нормальным и не трактуется как пропуск данных.